# Parse data
Since each example consists of 4 lines, we process 4 lines at a time. We preprocess the sentences, record the entity tag positions, and remove the tags. Then we extract the relationship information and add the data to the list.

For example, the train file.txt looks like below

_10	"The solute was placed inside a beaker and 5 mL of the <e1>solvent</e1> was pipetted into a 25 mL glass <e2>flask</e2> for each trial."_

_Entity-Destination(e1,e2)_

_Comment:_

_(empty)_




In [1]:
import re
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
from transformers import AutoTokenizer
from torch.utils.data import DataLoader, TensorDataset, Subset
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold
from torch.optim import AdamW
import matplotlib.pyplot as plt
import numpy as np
import torch.nn.functional as F
np.random.seed(42)


def parse_data(file_path, is_test=False):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    if is_test:
        for line in lines:
            parts = line.split('\t')
            sentence = parts[1].strip()
            sentence = sentence.replace('< e1 >', '<e1>')
            sentence = sentence.replace('< e2 >', '<e2>')
            sentence = sentence.replace('< /e1 >', '</e1>')
            sentence = sentence.replace('< /e2 >', '</e2>')

            sentence = re.sub(' {2,}', ' ', sentence)
            e1_start = sentence.find('<e1>')
            e1_end = sentence.find('</e1>')
            e2_start = sentence.find('<e2>')
            e2_end = sentence.find('</e2>')

            sentence = sentence.replace('<e1>', '').replace('</e1>', '').replace('<e2>', '').replace('</e2>', '')

            data.append({
                'sentence': sentence.strip('\"'),
                'e1_start': e1_start,
                'e1_end': e1_end - 4,
                'e2_start': e2_start - 4 if e2_start > e1_end else e2_start - 8,
                'e2_end': e2_end - 4 if e2_end > e1_end else e2_end - 8,
                'relation': None  # 테스트 데이터에는 레이블이 없으므로 None으로 설정
            })
    else:
        for i in range(0, len(lines), 4):  # Each example spans 4 lines
            sentence = lines[i].split('\t')[1].strip()
            sentence = sentence.replace('< e1 >', '<e1>')
            sentence = sentence.replace('< e2 >', '<e2>')
            sentence = sentence.replace('< /e1 >', '</e1>')
            sentence = sentence.replace('< /e2 >', '</e2>')

            sentence = re.sub(' {2,}', ' ', sentence)
            e1_start = sentence.find('<e1>')
            e1_end = sentence.find('</e1>')
            e2_start = sentence.find('<e2>')
            e2_end = sentence.find('</e2>')

            sentence = sentence.replace('<e1>', '').replace('</e1>', '').replace('<e2>', '').replace('</e2>', '')

            relation = lines[i + 1].strip()

            data.append({
                'sentence': sentence.strip('\"'),
                'e1_start': e1_start,
                'e1_end': e1_end - 4,
                'e2_start': e2_start - 4 if e2_start > e1_end else e2_start - 8,
                'e2_end': e2_end - 4 if e2_end > e1_end else e2_end - 8,
                'relation': relation
            })

    return data

In [3]:
valid_data = parse_data('TEST_FILE_FULL.TXT')
all_labels = set([item['relation'] for item in valid_data])
label_map = {label: idx for idx, label in enumerate(sorted(all_labels))}

In [4]:
def preprocess_data(data, tokenizer, label_map):
    processed_data = []

    for item in data:
        sentence = item['sentence']

        e1_start, e1_end = item['e1_start'], item['e1_end']
        e2_start, e2_end = item['e2_start'], item['e2_end']


        sentence = (
            sentence[:e1_start] + '[E1]' + sentence[e1_start:e1_end] + '[/E1]' +
            sentence[e1_end:e2_start] + '[E2]' + sentence[e2_start:e2_end] + '[/E2]' +
            sentence[e2_end:]
        )

        # Tokenize the sentence
        inputs = tokenizer(sentence, padding='max_length', truncation=True, max_length=128, return_tensors='pt')

        # Convert label to ID
        label = label_map[item['relation']]

        processed_data.append({
            'input_ids': inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0),
            'label': label
        })

    return processed_data

In [5]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

valid_data = parse_data('TEST_FILE_FULL.TXT')
test_processed = preprocess_data(valid_data, tokenizer, label_map)
input_ids = torch.stack([item['input_ids'] for item in test_processed])
attention_masks = torch.stack([item['attention_mask'] for item in test_processed])
labels_test = torch.tensor([item['label'] for item in test_processed])

valid_dataset = TensorDataset(input_ids, attention_masks, labels_test)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False)  # Don't shuffle test data

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [6]:
def load_glove_embeddings(glove_path, embedding_dim):
    glove_dict = {}
    with open(glove_path, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.strip().split()
            word = values[0]
            vector = np.asarray(values[1:], dtype='float32')
            glove_dict[word] = vector
    return glove_dict

# Load 300-dimensional embeddings
glove_path = "glove.6B.300d.txt"
glove_dict = load_glove_embeddings(glove_path, embedding_dim=300)

In [7]:
def create_embedding_matrix(tokenizer, glove_dict, embedding_dim):
    vocab_size = len(tokenizer)
    embedding_matrix = np.random.uniform(-0.05, 0.05, (vocab_size, embedding_dim))  # Random initialization

    for word, idx in tokenizer.vocab.items():
        if word in glove_dict:
            embedding_matrix[idx] = glove_dict[word]  # Use pre-trained embedding

    return torch.tensor(embedding_matrix, dtype=torch.float32)

# Create the embedding matrix
embedding_matrix = create_embedding_matrix(tokenizer, glove_dict, embedding_dim=300)

In [8]:
class BiLSTMAttention(nn.Module):
    '''
    A Bidirectional LSTM (BiLSTM) model with Attention mechanism for text classification.

    Args:
        vocab_size (int): Size of the vocabulary.
        embedding_dim (int): Dimensionality of word embeddings.
        embedding_matrix (torch.Tensor): Pre-trained word embedding matrix.
        hidden_dim (int): Number of hidden units in the LSTM.
        num_classes (int): Number of output classes for classification.
        dropout_rate (float, optional): Dropout rate for regularization. Defaults to 0.3.

    Methods:
        forward(input_ids, attention_mask=None, labels=None):
            Performs a forward pass through the model.

            Args:
                input_ids (torch.Tensor): Tensor containing tokenized input sequences.
                attention_mask (torch.Tensor, optional): Tensor indicating which tokens should be attended to. Defaults to None.
                labels (torch.Tensor, optional): Tensor containing the ground truth labels. Defaults to None.

            Returns:
                dict: A dictionary containing:
                    - 'logits' (torch.Tensor): The predicted class scores.
                    - 'loss' (torch.Tensor or None): The classification loss if labels are provided, otherwise None.
    '''
    def __init__(self, vocab_size, embedding_dim, embedding_matrix, hidden_dim, num_classes, dropout_rate=0.3):
        super(BiLSTMAttention, self).__init__()
        self.embedding = nn.Embedding.from_pretrained(embedding_matrix, freeze=False, padding_idx=0)
        self.lstm = nn.LSTM(input_size=embedding_dim, hidden_size=hidden_dim, num_layers=1, bidirectional=True, batch_first=True)
        self.dropout_lstm = nn.Dropout(dropout_rate)
        self.attention = nn.Linear(hidden_dim*2, 1)
        self.fc = nn.Linear(hidden_dim*2, num_classes)
        self.dropout_fc = nn.Dropout(dropout_rate)
    def forward(self, input_ids, attention_mask=None, labels=None):
        embedded = self.embedding(input_ids)
        lstm_out, _ = self.lstm(embedded)
        lstm_out = self.dropout_lstm(lstm_out)

        attention_scores = self.attention(lstm_out).squeeze(-1)
        attention_weights = torch.softmax(attention_scores, dim=1)
        attention_output = torch.sum(lstm_out * attention_weights.unsqueeze(-1), dim=1)
        attention_output = self.dropout_fc(attention_output)

        logits = self.fc(attention_output)
        loss = None
        if labels != None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)
        return {'logits': logits, 'loss': loss}

In [14]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = BiLSTMAttention(vocab_size=len(tokenizer), embedding_dim=300, embedding_matrix=embedding_matrix, hidden_dim=256, num_classes=len(label_map), dropout_rate=0.2).to(device)

model.load_state_dict(torch.load('best_model_custom_lstm.pth', weights_only=True, map_location=device))

model.eval()



predictions, true_labels = [], []
with torch.no_grad():
  for batch in valid_loader:
      input_ids, attention_mask, labels = batch
      input_ids = input_ids.to(device)
      attention_mask = attention_mask.to(device)
      labels = labels.to(device)

      outputs = model(input_ids, attention_mask=attention_mask)
      logits = outputs['logits']

      preds = torch.argmax(logits, dim=1)

      predictions.extend(preds.cpu().numpy())
      true_labels.extend(labels.cpu().numpy())

  accuracy = accuracy_score(true_labels, predictions)
  predictions = np.array(predictions)
  true_labels = np.array(true_labels)
  mask = true_labels != label_map['Other']
  filtered_predictions = predictions[mask]
  filtered_true_labels = true_labels[mask]
  f1 = f1_score(filtered_true_labels, filtered_predictions, average='macro')
  print(f'Validation Accuracy: {accuracy*100}%')
  print(f'F1 score without the relation other is: {f1}')

Validation Accuracy: 69.12035333087965%
F1 score without the relation other is: 0.6771612119021284
